In [2]:
import pandas as pd
import matplotlib.pyplot as plt

1. Inspect data

In [3]:
df = pd.read_csv('machine_log.csv')

In [4]:
df.shape

(2175, 6)

In [5]:
df.dtypes

timestamp             str
machine               str
temperature_C     float64
vibration_mm_s    float64
power_kW          float64
defects             int64
dtype: object

In [6]:
df.isna().sum()

timestamp          0
machine            0
temperature_C     40
vibration_mm_s     0
power_kW           0
defects            0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(15)

In [8]:
df.head()

,timestamp,machine,temperature_C,vibration_mm_s,power_kW,defects
0,2026-09-01 00:00:00,M1,69.279548,1.896258,14.777368,4
1,2026-09-01 01:00:00,M1,69.362059,1.101588,15.664637,2
2,2026-09-01 02:00:00,M1,68.915207,2.069531,15.464595,5
3,2026-09-01 03:00:00,M1,73.283183,2.066304,13.523357,2
4,2026-09-01 04:00:00,M1,71.231711,2.786648,15.083314,4


2. Clean data

In [9]:
df.columns

Index(['timestamp', 'machine', 'temperature_C', 'vibration_mm_s', 'power_kW',
       'defects'],
      dtype='str')

In [10]:
df = df.drop_duplicates()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['temperature_C'] = df['temperature_C'].fillna(df.groupby('machine')['temperature_C'].transform('median'))

3. Status labels (plain Python logic)


In [11]:
def get_status(temp, vib):
    if temp > 80:
        return "Overheat"
    elif vib > 3.0:
        return "High Vibration"
    else:
        return "OK"

In [12]:
df['status'] = df.apply(
    lambda row: get_status(row['temperature_C'], row['vibration_mm_s']), axis=1)

4. Filter and GroupBy



In [15]:
summary = df.groupby("machine").agg(
    mean_temp=("temperature_C", "mean"),
    mean_vib=("vibration_mm_s", "max"),
    total_defects=("defects", "sum"),
    overheat_count=("status", lambda x: (x == "Overheat").sum()),
)

In [16]:
summary["total_defects"].idxmax()   # machine with most defects
summary["overheat_count"].idxmax()   # machine that overheats most

'M3'

In [17]:
summary.columns

Index(['mean_temp', 'mean_vib', 'total_defects', 'overheat_count'], dtype='str')